In [1]:

import sys
sys.path.insert(0, '/n/home12/binxuwang/Github/AccentuationPredRMT')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print("imports ok")


imports ok


In [2]:

# ── Section 1: Theory background ──
# Just a markdown cell in the notebook, executed here as a comment
"""
# RMT Theory Validation: Ridge Regression Weight Deviation & Accentuation Error

## Theory summary

For ridge regression β̂ = (XᵀX + nλI)⁻¹Xᵀy with x ~ N(0, Σ), the RMT
three-term formula predicts the per-PC squared error as n,d → ∞ with d/n = γ fixed:

  E[(uₖᵀ(β̂ - β*))²] ≈
      κ²/(λₖ+κ)² · (β*ᵀuₖ)²                     [Term 1: overshrinkage bias]
    + κ²λₖ/(λₖ+κ)² · C_sig / (n - df₂)           [Term 2: finite-sample signal]
    + σ²λₖ/(λₖ+κ)² / (n - df₂)                   [Term 3: finite-sample noise]

where κ = κ(λ) solves the fixed-point equation and df₂ = Σ_k λₖ²/(λₖ+κ)².

The accentuation alignment R_det ≈ β*ᵀ(Σ+κI)⁻¹Σβ* / [β*ᵀΣ²(Σ+κI)⁻²β* + (σ²/n)·Tr(Σ(Σ+κI)⁻²)]
"""
print("Section 1: Theory background")


Section 1: Theory background


In [3]:

# ── Section 2: Small-d per-PC error — three spectra ──
from rmt_core import get_spectrum, SpectrumKappa, ridge_error_per_pc_theory
from rmt_core.simulation_lib import run_monte_carlo

rng = np.random.default_rng(42)

configs = [
    dict(name='isotropic',   kwargs=dict(d=40),              n=100, lam=0.1, sigma=0.5),
    dict(name='powerlaw',    kwargs=dict(d=64, alpha=1.0),   n=200, lam=0.05, sigma=0.5),
    dict(name='vanhateren',  kwargs=dict(patch_hw=8, n_patches_per_image=200, n_images=50),
         n=500, lam=0.05, sigma=0.5),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
fig.suptitle('Per-PC Ridge Error: Theory vs Monte Carlo  (small d)', fontsize=13, y=1.01)

for row, cfg in enumerate(configs):
    rng_i = np.random.default_rng(42 + row)
    lam, sigma, n = cfg['lam'], cfg['sigma'], cfg['n']

    eigenvalues, eigenvectors = get_spectrum(cfg['name'], rng=rng_i, **cfg['kwargs'])
    d = len(eigenvalues)
    gamma = d / n
    beta_star = rng_i.standard_normal(d)
    beta_proj = eigenvectors.T @ beta_star

    kappa_fn = SpectrumKappa(eigenvalues, gamma)
    kappa = kappa_fn(lam)

    err_theory, t1, t2, t3 = ridge_error_per_pc_theory(eigenvalues, beta_proj, kappa, sigma, n)

    mc = run_monte_carlo(n, eigenvalues, eigenvectors, beta_star, sigma, lam,
                         n_trials=500, rng=rng_i)
    err_sim = mc['error_per_pc']

    label = f'{cfg["name"]}, d={d}, n={n}, γ={gamma:.2f}, λ={lam}'

    # Eigenspectrum
    ax = axes[row, 0]
    ax.semilogy(eigenvalues, '-o', ms=2, lw=1.2, color='steelblue')
    ax.set_xlabel('PC index k'); ax.set_ylabel('λ_k')
    ax.set_title(f'Spectrum ({cfg["name"]})')

    # Per-PC error
    ax = axes[row, 1]
    ax.semilogy(err_sim, 'o', ms=3, alpha=0.55, color='gray', label='MC')
    ax.semilogy(err_theory, '-', lw=1.8, color='C0', label='Theory')
    ax.semilogy(t1, '--', lw=1.1, color='C1', label='T1: bias')
    ax.semilogy(t2, '--', lw=1.1, color='C2', label='T2: signal')
    ax.semilogy(t3, '--', lw=1.1, color='C3', label='T3: noise')
    ax.set_xlabel('PC k'); ax.set_ylabel('E[(uₖᵀΔβ)²]')
    ax.set_title(label)
    ax.legend(fontsize=7)

    # Scatter
    ax = axes[row, 2]
    ax.loglog(err_theory, err_sim, 'o', ms=3, alpha=0.6, color='C0')
    lims = [min(err_theory.min(), err_sim.min()), max(err_theory.max(), err_sim.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel('Theory'); ax.set_ylabel('MC')
    ax.set_title(f'Scatter (gen err: th={np.sum(eigenvalues*err_theory):.3f}, mc={mc["total_gen_error"]:.3f})')

plt.tight_layout()
plt.savefig('/n/home12/binxuwang/Github/AccentuationPredRMT/figures/nb_small_d_per_pc.png',
            dpi=130, bbox_inches='tight')
plt.show()
print("done")


done


In [4]:

# ── Section 3: Large-d validation (GPU) ──
from rmt_core.gpu_simulation_lib import run_monte_carlo_gpu, get_device
from rmt_core.spectrum_lib import vanhateren_spectrum, powerlaw_spectrum

device = get_device()
print(f"Device: {device}")

large_d_configs = [
    dict(label='powerlaw d=1000 γ=1.0',  spectrum='powerlaw', d=1000, n=1000, lam=0.05, sigma=0.5, top_k=100, alpha=1.0),
    dict(label='powerlaw d=5000 γ=2.5',  spectrum='powerlaw', d=5000, n=2000, lam=0.05, sigma=0.5, top_k=150, alpha=1.0),
    dict(label='vanhateren 32×32 d=1024', spectrum='vanhateren', patch_hw=32, n=2000, lam=0.05, sigma=0.5, top_k=100),
    dict(label='vanhateren 100×100 d=10000', spectrum='vanhateren', patch_hw=100, n=5000, lam=0.05, sigma=0.5, top_k=200),
]

fig, axes = plt.subplots(len(large_d_configs), 3, figsize=(15, 4.5*len(large_d_configs)))
fig.suptitle('Per-PC Ridge Error: Large d (GPU validation)', fontsize=13, y=1.01)

for row, cfg in enumerate(large_d_configs):
    rng_i = np.random.default_rng(42 + row)
    n, lam, sigma, top_k = cfg['n'], cfg['lam'], cfg['sigma'], cfg['top_k']

    if cfg['spectrum'] == 'vanhateren':
        eigenvalues, eigenvectors_topk, _ = vanhateren_spectrum(
            patch_hw=cfg['patch_hw'], n_patches_per_image=300,
            n_images=80, use_randomized_svd=True, top_k=top_k, rng=rng_i)
        d_full = cfg['patch_hw'] ** 2
        gamma = top_k / n
        beta_star = rng_i.standard_normal(d_full)
        beta_proj_topk = eigenvectors_topk.T @ beta_star
        beta_proj_full = beta_proj_topk
        diagonal = False
        eigenvalues_data = eigenvalues[:top_k]
    else:
        eigenvalues, _ = powerlaw_spectrum(cfg['d'], alpha=cfg['alpha'])
        d_full = cfg['d']
        gamma = d_full / n
        eigenvectors_topk = np.eye(d_full, top_k)
        beta_star = rng_i.standard_normal(d_full)
        beta_proj_topk = eigenvectors_topk.T @ beta_star
        beta_proj_full = beta_star
        diagonal = True
        eigenvalues_data = eigenvalues  # full for diagonal sampling

    eigenvalues_topk = eigenvalues[:top_k]
    kappa_fn = SpectrumKappa(eigenvalues, gamma)
    kappa = kappa_fn(lam)

    err_theory, t1, t2, t3 = ridge_error_per_pc_theory(
        eigenvalues_topk, beta_proj_topk, kappa, sigma, n,
        eigenvalues_full=eigenvalues, beta_proj_full=beta_proj_full)

    mc = run_monte_carlo_gpu(
        n, eigenvalues_data, eigenvectors_topk, beta_star, sigma, lam,
        n_trials=100, diagonal=diagonal, device=device, rng=rng_i, verbose=False)
    err_sim = mc['error_per_pc']

    # Eigenspectrum
    ax = axes[row, 0]
    ax.semilogy(eigenvalues, '-', lw=1.2, color='steelblue')
    ax.set_xlabel('PC k'); ax.set_ylabel('λ_k')
    ax.set_title(f'Spectrum — {cfg["label"]}')
    ax.axvline(top_k, color='r', lw=0.8, linestyle=':', label=f'top-K={top_k}')
    ax.legend(fontsize=7)

    # Per-PC error
    ax = axes[row, 1]
    ax.semilogy(err_sim, 'o', ms=2.5, alpha=0.55, color='gray', label='MC (GPU)')
    ax.semilogy(err_theory, '-', lw=1.8, color='C0', label='Theory')
    ax.semilogy(t1, '--', lw=1, color='C1', label='T1: bias')
    ax.semilogy(t2, '--', lw=1, color='C2', label='T2: signal')
    ax.semilogy(t3, '--', lw=1, color='C3', label='T3: noise')
    ax.set_xlabel('PC k (top-K)'); ax.set_ylabel('E[(uₖᵀΔβ)²]')
    ax.set_title(f'{cfg["label"]}  κ={kappa:.4f}  γ={gamma:.3f}')
    ax.legend(fontsize=7)

    # Scatter
    ax = axes[row, 2]
    ax.loglog(err_theory, err_sim, 'o', ms=3, alpha=0.6, color='C0')
    lims = [min(err_theory.min(), err_sim.min()), max(err_theory.max(), err_sim.max())]
    ax.plot(lims, lims, 'k--', lw=1, label='y=x')
    ax.set_xlabel('Theory'); ax.set_ylabel('MC')
    gen_th = float(np.sum(eigenvalues_topk * err_theory))
    ax.set_title(f'gen: th={gen_th:.3f}, mc={mc["total_gen_error"]:.3f}')
    ax.legend(fontsize=7)

    print(f"  {cfg['label']}: theory={gen_th:.4f}, MC={mc['total_gen_error']:.4f}")

plt.tight_layout()
plt.savefig('/n/home12/binxuwang/Github/AccentuationPredRMT/figures/nb_large_d.png',
            dpi=130, bbox_inches='tight')
plt.show()
print("done")


Device: cuda
  powerlaw d=1000 γ=1.0: theory=5.6412, MC=5.6644
  powerlaw d=5000 γ=2.5: theory=63.3693, MC=62.7900
  Randomized SVD: d=1024, N=24000, k=100 ...
  vanhateren 32×32 d=1024: theory=0.6400, MC=0.6414
  Randomized SVD: d=10000, N=24000, k=200 ...
  vanhateren 100×100 d=10000: theory=0.2142, MC=0.2142
done


In [5]:

# ── Section 4: σ sweep — powerlaw and vanhateren ──
import os

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Theory vs MC as a function of noise σ  (d=256, n=512, λ=0.05)', fontsize=12)

sweep_files = [
    ('powerlaw d=256', '/n/home12/binxuwang/Github/AccentuationPredRMT/tables/sigma_sweep_powerlaw_d256_n512_lam0.05.npz'),
    ('vanhateren 16×16', '/n/home12/binxuwang/Github/AccentuationPredRMT/tables/sigma_sweep_vanhateren_d256_n512_lam0.05.npz'),
]

for row, (label, fpath) in enumerate(sweep_files):
    data = np.load(fpath)
    sigma_vals = data['sigma']
    th_gen  = data['theory_gen'];  mc_gen  = data['mc_gen']
    th_acc  = data['theory_acc'];  mc_acc  = data['mc_acc']
    th_R    = data['theory_R'];    mc_R    = data['mc_R']

    # (a) Generalization error
    ax = axes[row, 0]
    ax.plot(sigma_vals, th_gen, '-o', lw=1.8, label='Theory', color='C0')
    ax.plot(sigma_vals, mc_gen, '--s', lw=1.5, label='MC', color='C1')
    ax.set_xlabel('σ'); ax.set_ylabel('Total gen error')
    ax.set_title(f'{label} — Generalization error')
    ax.legend()

    # (b) Accentuation error
    ax = axes[row, 1]
    ax.plot(sigma_vals, th_acc, '-o', lw=1.8, label='Theory', color='C0')
    ax.plot(sigma_vals, mc_acc, '--s', lw=1.5, label='MC', color='C1')
    ax.set_xlabel('σ'); ax.set_ylabel('Accentuation error')
    ax.set_title(f'{label} — Accentuation error')
    ax.legend()

    # (c) Alignment R
    ax = axes[row, 2]
    ax.plot(sigma_vals, th_R, '-o', lw=1.8, label='R_det (theory)', color='C0')
    ax.plot(sigma_vals, mc_R, '--s', lw=1.5, label='R (MC)', color='C1')
    ax.axhline(1.0, color='gray', lw=0.8, linestyle=':')
    ax.set_xlabel('σ'); ax.set_ylabel('Alignment R')
    ax.set_title(f'{label} — Accentuation alignment')
    ax.legend()

plt.tight_layout()
plt.savefig('/n/home12/binxuwang/Github/AccentuationPredRMT/figures/nb_sigma_sweep.png',
            dpi=130, bbox_inches='tight')
plt.show()
print("done")


done


In [6]:

# ── Section 5: λ sweep — theory predictions for varying ridge penalty ──
from rmt_core import accentuation_error_theory, ridge_error_total_theory, SpectrumKappa
from rmt_core.spectrum_lib import powerlaw_spectrum

rng_lam = np.random.default_rng(99)

d, n, sigma = 256, 512, 1.0
eigenvalues_pl, _ = powerlaw_spectrum(d, alpha=1.0)
gamma = d / n
beta_star = rng_lam.standard_normal(d)
beta_proj = beta_star  # diagonal

lam_vals = np.geomspace(1e-3, 10, 40)
gen_errors_th = []
acc_errors_th = []
R_vals_th = []

for lam in lam_vals:
    kappa_fn = SpectrumKappa(eigenvalues_pl, gamma)
    kappa = kappa_fn(lam)
    gen_err = ridge_error_total_theory(eigenvalues_pl, beta_proj, kappa, sigma, n)
    acc_err, R_det, _ = accentuation_error_theory(eigenvalues_pl, beta_proj, kappa, sigma, n)
    gen_errors_th.append(gen_err)
    acc_errors_th.append(acc_err)
    R_vals_th.append(R_det)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Theory predictions vs ridge penalty λ  (powerlaw d={d}, n={n}, σ={sigma})', fontsize=11)

ax = axes[0]
ax.loglog(lam_vals, gen_errors_th, '-o', ms=3, lw=1.5, color='C0')
ax.set_xlabel('λ (ridge penalty)'); ax.set_ylabel('Total gen error')
ax.set_title('Generalization error vs λ')
ax.axvline(0.05, color='r', lw=0.8, linestyle='--', label='λ=0.05 (sweep)')
ax.legend(fontsize=8)

ax = axes[1]
ax.semilogx(lam_vals, acc_errors_th, '-o', ms=3, lw=1.5, color='C1')
ax.set_xlabel('λ'); ax.set_ylabel('Accentuation error')
ax.set_title('Accentuation error vs λ')
ax.axvline(0.05, color='r', lw=0.8, linestyle='--')

ax = axes[2]
ax.semilogx(lam_vals, R_vals_th, '-o', ms=3, lw=1.5, color='C2')
ax.axhline(1.0, color='gray', lw=0.8, linestyle=':')
ax.set_xlabel('λ'); ax.set_ylabel('R_det (alignment)')
ax.set_title('Accentuation alignment vs λ')
ax.axvline(0.05, color='r', lw=0.8, linestyle='--')

plt.tight_layout()
plt.savefig('/n/home12/binxuwang/Github/AccentuationPredRMT/figures/nb_lambda_sweep.png',
            dpi=130, bbox_inches='tight')
plt.show()
print("done")


done
